<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.2-custom-fastapi/notebooks/GCP_Capstone_11.2_CustomFastAPI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.2 Custom FastAPI + vLLM — Production Inference Server
**Netsetos GenAI Engineering — GCP Capstone**

Wrap vLLM in FastAPI for per-tenant auth, SSE streaming, BigQuery logs, PII redaction, guided JSON output.


## Cell 1: FastAPI App with Lifespan Engine Loading


In [ ]:
MAIN_PY = '''
# main.py - Production FastAPI + vLLM server
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request, HTTPException, Depends, BackgroundTasks, Security
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.security import APIKeyHeader
from fastapi.exceptions import RequestValidationError
from slowapi import Limiter
from slowapi.errors import RateLimitExceeded
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams
from vllm.utils import random_uuid
import asyncio, json, time, datetime, os, logging

logger = logging.getLogger("documind")

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Loading model into GPU...")
    engine_args = AsyncEngineArgs(
        model=os.getenv("MODEL_NAME", "google/gemma-3-4b-it"),
        tensor_parallel_size=1,
        gpu_memory_utilization=0.90,
        max_model_len=8192,
        dtype="auto",
        trust_remote_code=True,
    )
    app.state.engine = AsyncLLMEngine.from_engine_args(engine_args)
    app.state.engine_ready = True
    logger.info("Engine loaded. GPU ready.")
    yield
    app.state.engine_ready = False
    app.state.engine = None

app = FastAPI(title="DocuMind AI Inference", version="1.0.0", lifespan=lifespan)
'''
with open('main.py', 'w') as f:
    f.write(MAIN_PY)
print('main.py (part 1 of 4) written')
print('Lifespan loads AsyncLLMEngine on startup, releases on shutdown')


## Cell 2: Pydantic v2 Schema (OpenAI-Compatible)


In [ ]:
SCHEMA_PY = '''
# schemas.py - Pydantic v2 models matching OpenAI API
from typing import Literal, Optional, Union
from pydantic import BaseModel, Field
import time, uuid

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant", "tool"]
    content: Optional[str] = None
    name: Optional[str] = None

class ChatCompletionRequest(BaseModel):
    model: str
    messages: list[ChatMessage]
    temperature: Optional[float] = Field(default=1.0, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=1.0, ge=0.0, le=1.0)
    max_tokens: Optional[int] = None
    stream: Optional[bool] = False
    stop: Optional[Union[str, list[str]]] = None
    user: Optional[str] = None
    model_config = {"extra": "allow"}

class AssistantMessage(BaseModel):
    role: Literal["assistant"] = "assistant"
    content: Optional[str] = None

class UsageInfo(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class ChatCompletionChoice(BaseModel):
    index: int
    message: AssistantMessage
    finish_reason: Optional[Literal["stop","length","tool_calls","content_filter"]] = None

class ChatCompletionResponse(BaseModel):
    id: str = Field(default_factory=lambda: f"chatcmpl-{uuid.uuid4().hex[:29]}")
    object: Literal["chat.completion"] = "chat.completion"
    created: int = Field(default_factory=lambda: int(time.time()))
    model: str
    choices: list[ChatCompletionChoice]
    usage: UsageInfo

# Streaming delta model
class DeltaMessage(BaseModel):
    role: Optional[Literal["assistant"]] = None
    content: Optional[str] = None

class ChatCompletionStreamChoice(BaseModel):
    index: int
    delta: DeltaMessage
    finish_reason: Optional[str] = None

class ChatCompletionChunk(BaseModel):
    id: str
    object: Literal["chat.completion.chunk"] = "chat.completion.chunk"
    created: int
    model: str
    choices: list[ChatCompletionStreamChoice]
'''
with open('schemas.py', 'w') as f:
    f.write(SCHEMA_PY)
print('schemas.py written')
print('Pydantic v2 models matching OpenAI Chat Completions API exactly')


## Cell 3: SSE Streaming Generator


In [ ]:
STREAMING_PY = '''
import asyncio, json, time
from fastapi import Request

async def generate_sse_stream(engine, request_id, model_name, prompt, sampling_params, request: Request):
    """Convert vLLM RequestOutput stream to OpenAI SSE format."""
    created = int(time.time())
    first_chunk = True
    prev_len = {}  # completion index -> chars already sent
    try:
        async for output in engine.generate(prompt, sampling_params, request_id):
            # Client disconnected? Abort and free GPU
            if await request.is_disconnected():
                await engine.abort(request_id)
                return
            
            for completion in output.outputs:
                if first_chunk:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0,
                                "delta": {"role": "assistant", "content": ""},
                                "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
                    first_chunk = False
                
                # vLLM completion.text is CUMULATIVE -> emit only the new suffix
                idx = completion.index
                sent = prev_len.get(idx, 0)
                new_text = completion.text[sent:]
                prev_len[idx] = len(completion.text)
                if new_text:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0,
                                "delta": {"content": new_text},
                                "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
                
                if completion.finish_reason is not None:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0, "delta": {},
                                "finish_reason": completion.finish_reason}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
        
        yield "data: [DONE]\\n\\n"
    except asyncio.CancelledError:
        await engine.abort(request_id)
        raise
'''
with open('streaming.py', 'w') as f:
    f.write(STREAMING_PY)
print('streaming.py written')
print('CRITICAL: SSE headers needed when mounting:')
print('  Content-Type: text/event-stream')
print('  Cache-Control: no-cache')
print('  X-Accel-Buffering: no  (disable proxy buffering)')


## Cell 4: Auth + Rate Limiting


In [ ]:
AUTH_PY = '''
import hashlib
import datetime
from fastapi import Depends, HTTPException, Security, Request
from fastapi.security import APIKeyHeader
from slowapi import Limiter
from google.cloud import firestore

api_key_header = APIKeyHeader(name="X-API-Key")
firestore_client = firestore.Client()

def hash_api_key(plaintext: str) -> str:
    """SHA-256 for lookup; for production use bcrypt with salt."""
    return hashlib.sha256(plaintext.encode()).hexdigest()

async def get_tenant(api_key: str = Security(api_key_header)) -> dict:
    """Lookup tenant by HASHED key. Supports key rotation via multiple active keys."""
    key_hash = hash_api_key(api_key)
    doc = firestore_client.collection("api_keys").document(key_hash).get()
    if not doc.exists:
        raise HTTPException(status_code=401, detail="Invalid API key")
    
    data = doc.to_dict()
    # Check expiry for key rotation
    if data.get("expires_at") and data["expires_at"] < datetime.datetime.utcnow():
        raise HTTPException(status_code=401, detail="Expired API key")
    
    # Fetch tenant details
    tenant_doc = firestore_client.collection("tenants").document(data["tenant_id"]).get()
    return tenant_doc.to_dict()  # {tenant_id, tier, rate_limit}

def rate_limit_key(request: Request):
    return request.headers.get("X-API-Key", "anonymous")

limiter = Limiter(key_func=rate_limit_key)

def tier_rate_limit(request: Request):
    tier = getattr(request.state, "tenant_tier", "free")
    return {"free": "10/minute",
            "pro": "100/minute",
            "enterprise": "1000/minute"}.get(tier, "10/minute")
'''
with open('auth.py', 'w') as f:
    f.write(AUTH_PY)
print('auth.py written')
print('API keys HASHED (never plaintext) in Firestore')
print('Multiple active keys per tenant with expiry for rotation')
print('Tiered rate limits: free 10/min, pro 100/min, enterprise 1000/min')


## Cell 5: PII Redaction + BigQuery Logging


In [ ]:
LOGGING_PY = '''
import re, datetime
from google.cloud import bigquery
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

pii_analyzer = AnalyzerEngine()
pii_anonymizer = AnonymizerEngine()
bq_client = bigquery.Client()
BQ_TABLE = "project.dataset.inference_logs"

def redact_pii(text: str) -> str:
    """Presidio NER + regex fallback."""
    try:
        results = pii_analyzer.analyze(text=text, language="en")
        if results:
            return pii_anonymizer.anonymize(text=text, analyzer_results=results).text
    except Exception:
        pass
    # Regex safety net
    text = re.sub(r"\\b[\\w.-]+@[\\w.-]+\\.\\w+\\b", "<EMAIL>", text)
    text = re.sub(r"\\b\\d{3}[-.]?\\d{3}[-.]?\\d{4}\\b", "<PHONE>", text)
    text = re.sub(r"\\b\\d{3}-\\d{2}-\\d{4}\\b", "<SSN>", text)
    return text

async def log_request(entry: dict):
    """Non-blocking BigQuery streaming insert."""
    errors = bq_client.insert_rows_json(BQ_TABLE, [entry])
    if errors:
        print(f"BQ insert failed: {errors}")
'''
with open('logging_module.py', 'w') as f:
    f.write(LOGGING_PY)
print('logging_module.py written')
print()
print('BigQuery table schema:')
print('  CREATE TABLE inference_logs (')
print('    request_id STRING, tenant_id STRING, model STRING,')
print('    prompt_tokens INT64, completion_tokens INT64,')
print('    latency_ms FLOAT64, status_code INT64,')
print('    timestamp TIMESTAMP, prompt_preview STRING')
print('  ) PARTITION BY DATE(timestamp)')
print('    CLUSTER BY tenant_id, model;')


## Cell 6: Custom DocuMind Endpoints with Guided JSON


In [ ]:
DOCUMIND_PY = '''
from pydantic import BaseModel
from typing import Literal
from fastapi import APIRouter, Request, Depends
from vllm.sampling_params import SamplingParams, StructuredOutputsParams
from vllm.utils import random_uuid
import json

router = APIRouter(prefix="/v1/documind", tags=["documind"])

class ClassificationResult(BaseModel):
    category: Literal["invoice","contract","report","letter","other"]
    confidence: float
    reasoning: str

# Request bodies: a scalar param binds as a QUERY string, but clients POST JSON.
# Wrap inputs in a model so FastAPI reads them from the request body.
class ClassifyRequest(BaseModel):
    text: str

class ExtractRequest(BaseModel):
    text: str
    schema_def: dict  # client-provided JSON Schema

@router.post("/classify")
async def classify_document(req: ClassifyRequest, request: Request, tenant: dict = Depends(get_tenant)):
    so = StructuredOutputsParams(json=ClassificationResult.model_json_schema())
    sampling = SamplingParams(temperature=0.0, max_tokens=256, structured_outputs=so)
    prompt = f"Classify this document into: invoice, contract, report, letter, other.\\n\\nDocument:\\n{req.text[:4000]}\\n\\nRespond with JSON:"
    
    request_id = random_uuid()
    final = None
    async for output in request.app.state.engine.generate(prompt, sampling, request_id):
        final = output
    # Guided decoding GUARANTEES valid JSON matching schema
    return ClassificationResult.model_validate_json(final.outputs[0].text)

@router.post("/extract")
async def extract_fields(req: ExtractRequest, request: Request, tenant: dict = Depends(get_tenant)):
    so = StructuredOutputsParams(json=req.schema_def)
    sampling = SamplingParams(temperature=0.0, max_tokens=1024, structured_outputs=so)
    prompt = f"Extract structured data matching the schema.\\n\\nDocument:\\n{req.text[:4000]}\\n\\nJSON:"
    
    request_id = random_uuid()
    final = None
    async for output in request.app.state.engine.generate(prompt, sampling, request_id):
        final = output
    return json.loads(final.outputs[0].text)
'''
with open('documind.py', 'w') as f:
    f.write(DOCUMIND_PY)
print('documind.py written')
print()
print('/v1/documind/classify - 100% valid JSON matching ClassificationResult')
print('/v1/documind/extract - 100% valid JSON matching client schema')
print('NO retry-on-parse-error logic needed &mdash; structured outputs enforce at token level')


## Cell 7: Dockerfile + Requirements


In [ ]:
DOCKERFILE = '''
FROM vllm/vllm-openai:v0.28.0

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

ENV PORT=8080
EXPOSE 8080

# --workers 1 MANDATORY - GPU memory not shareable across processes
CMD ["python", "-m", "uvicorn", "main:app", \\
     "--host", "0.0.0.0", "--port", "8080", \\
     "--workers", "1", "--timeout-keep-alive", "120"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)

REQUIREMENTS = '''
fastapi>=0.135.0
uvicorn[standard]>=0.34.0
slowapi>=0.1.9
pydantic>=2.8.0
google-cloud-bigquery>=3.25.0
google-cloud-firestore>=2.16.0
google-cloud-logging>=3.11.0
google-cloud-secret-manager>=2.20.0
opentelemetry-sdk>=1.27.0
opentelemetry-exporter-gcp-trace>=1.7.0
opentelemetry-instrumentation-fastapi>=0.48b0
presidio-analyzer>=2.2.0
presidio-anonymizer>=2.2.0
httpx>=0.27.0
'''
with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)

print('Dockerfile + requirements.txt written')
print('Base: vllm/vllm-openai:v0.28.0 (CUDA + PyTorch + vLLM)')
print('+ FastAPI + google-cloud + Presidio + OpenTelemetry')
print('Final image size: ~14GB')
print()
print('Deploy with:')
print('  gcloud run deploy documind-inference --source . \\\\')
print('    --gpu 1 --gpu-type nvidia-l4 --cpu 8 --memory 32Gi \\\\')
print('    --concurrency 10 --max-instances 3 --min-instances 1 \\\\')
print('    --no-cpu-throttling --no-gpu-zonal-redundancy')


## Cell 8: Client Code — OpenAI SDK + Custom httpx


In [ ]:
CLIENT_PY = '''
from openai import OpenAI
import httpx
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

SERVICE_URL = "https://documind-inference-xxxxx.run.app"
API_KEY = "sk-tenant-abc123"

# Standard OpenAI SDK works unchanged
client = OpenAI(
    base_url=f"{SERVICE_URL}/v1",
    api_key=API_KEY,
    default_headers={"X-API-Key": API_KEY},
)

# Non-streaming
response = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=[{"role": "user", "content": "Summarize this contract..."}],
    max_tokens=500,
)
print(response.choices[0].message.content)

# Streaming
stream = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=[{"role": "user", "content": "Explain this invoice..."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

# Custom DocuMind endpoint with cold start retry
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    retry=retry_if_exception_type((httpx.ConnectError, httpx.ReadTimeout)),
)
async def classify_document(text: str) -> dict:
    async with httpx.AsyncClient(timeout=120.0) as client:
        response = await client.post(
            f"{SERVICE_URL}/v1/documind/classify",
            headers={"X-API-Key": API_KEY},
            json={"text": text},
        )
        if response.status_code == 503:
            raise httpx.ReadTimeout("Cold start - retrying")
        response.raise_for_status()
        return response.json()
'''
with open('client.py', 'w') as f:
    f.write(CLIENT_PY)
print('client.py written')
print()
print('Standard openai SDK works unchanged - just change base_url')
print('For custom endpoints: httpx + tenacity for retry-on-cold-start')
print('Auth: X-API-Key header (custom) + Authorization: Bearer (OpenAI compat)')


## Done!
Complete production inference server:
- FastAPI lifespan loads AsyncLLMEngine (Pattern C)
- Pydantic v2 OpenAI-compatible schemas
- SSE streaming with client-disconnect handling
- Per-tenant auth with Firestore hashed keys
- SlowAPI tiered rate limiting (free/pro/enterprise)
- BigQuery logging with Presidio PII redaction
- Custom /v1/documind/classify + /extract with guided JSON
- Dockerfile + Cloud Run deploy command
- OpenAI SDK + httpx client patterns
